<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_0_Classification_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Foundations: Credit Default Prediction

**Author:** Brad Sheese

---

## Introduction

Classification is about predicting discrete categories. Unlike regression (predicting continuous values like price), classification places outcomes into categorical bins (defaulted vs. paid, spam vs. not spam, tumor vs. benign).

**Key distinction**: Classification models output probabilities (0-1), which we convert to hard predictions using a **threshold** (usually 0.5).

In this notebook, we explore classification concepts using the Credit Card Default dataset. This dataset is imbalanced (~22% default), making it ideal for understanding classification pitfalls.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Load and explore a classification dataset
2. Understand features and target variable relationship
3. Apply a basic classifier (Logistic Regression)
4. Interpret probability outputs and decision thresholds

## Setup and Data Loading

We'll use the Credit Card Default dataset from UCI ML Repository (via OpenML).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Load the Credit Card Default dataset from OpenML
from sklearn.datasets import fetch_openml

print("Loading dataset...")
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['class'].value_counts())
print(f"\nFirst few rows:")
df.head()

## Data Exploration

Let's understand the dataset structure. Notice the **imbalance**: there are more 'good' credits than 'bad' ones.

In [ ]:
# Check target variable
print("Target variable 'class' distribution:")
print(df['class'].value_counts())
print(f"\nPercentage of 'bad' (default) credits: {(df['class'] == 'bad').mean()*100:.1f}%")

# Check feature types
print(f"\nFeature dtypes summary:")
print(df.dtypes.value_counts())

# Check for missing values
print(f"\nMissing values: {df.isnull().sum().sum()}")

In [ ]:
# Visualize the class imbalance
plt.figure(figsize=(8, 5))
df['class'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Target Variable Distribution (Class Imbalance)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.annotate(f'\n imbalance:\n Only {(df["class"] == "bad").mean()*100:.1f}% are defaults', 
             xy=(0.5, 0.6), xycoords='axes fraction', ha='center')
plt.tight_layout()
plt.show()

## Data Preprocessing

We need to convert categorical features to numeric and prepare X (features) and y (target).

In [ ]:
# Separate features and target
# Convert target to binary: good = 0, bad = 1
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols[:5]}...")
print(f"Numerical columns ({len(num_cols)}): {num_cols[:5]}...")

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"\nFeatures after encoding: {X_encoded.shape[1]} columns")
X_encoded.head()

In [ ]:
# Train-test split with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

## Building a Basic Classifier: Logistic Regression

Logistic Regression outputs probabilities (0-1) using the sigmoid function.

Key formula: P(Y=1) = 1 / (1 + e^-(b0 + b1*X1 + ...))

In [ ]:
# Scale features (important for LogisticRegression to converge properly)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probability of default (class=1)

print("Model trained successfully!")
print(f"\nFirst 10 predictions (actual vs predicted):")
for i in range(10):
    print(f"  Actual: {y_test.iloc[i]:d}, Predicted: {y_pred[i]}, Probability: {y_proba[i]:.3f}")

## Interpreting Probabilities and the Default Threshold (0.5)

By default, Logistic Regression classifies as '1' (default) if probability >= 0.5, otherwise '0' (good).

This threshold of 0.5 is arbitrary - we'll explore how changing it affects results.

In [ ]:
# Examine prediction distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(y_proba, bins=50, edgecolor='black')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
plt.xlabel('Predicted Probability of Default')
plt.ylabel('Count')
plt.title('Distribution of Predicted Probabilities')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(y_proba[y_test==0], bins=30, alpha=0.7, label='Actual: Good')
plt.hist(y_proba[y_test==1], bins=30, alpha=0.7, label='Actual: Bad')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
plt.xlabel('Predicted Probability of Default')
plt.ylabel('Count')
plt.title('Probabilities by Actual Class')
plt.legend()

plt.tight_layout()
plt.show()

print(f"\nBasic Accuracy with default (0.5) threshold: {(y_pred == y_test).mean()*100:.2f}%")

## Key Takeaways

1. **Soft Predictions**: The probability output (0.0-1.0)
2. **Hard Predictions**: The binary classification (0 or 1) after applying threshold
3. **Default Threshold**: 0.5 is standard but not always optimal
4. **Class Imbalance**: Our dataset is ~22% defaults - this affects which metrics matter

In Notebook 2, we'll explore how to properly evaluate this model using the confusion matrix.